# Start code

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import json
import cv2
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create Torch Dataset

In [3]:
class KeypointsDataset(Dataset):
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, "r") as f:
            self.data = json.load(f)
        
        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
        h,w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(img)
        kps = np.array(item['kps']).flatten()
        kps = kps.astype(np.float32)

        kps[::2] *= 224.0 / w # Adjust x coordinates
        kps[1::2] *= 224.0 / h # Adjust y coordinates

        return img, kps

In [4]:

train_dataset = KeypointsDataset("/home/gcartasso/court points/data/images/","/home/gcartasso/court points/data/data_train.json")
val_dataset = KeypointsDataset("/home/gcartasso/court points/data/images/","/home/gcartasso/court points/data/data_val.json")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)


# Creat model

In [5]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small

# 1. Load pre-trained MobileNetV3-Small
model = mobilenet_v3_small(pretrained=True)

num_ftrs = model.classifier[-1].in_features  # Get input features of the last layer
model.classifier = nn.Sequential(
    *list(model.classifier.children())[:-1],  # Remove the original last layer
    nn.Linear(num_ftrs, 128),  # Optional: Add an intermediate layer
    nn.ReLU(),  # Activation (helps in regression)
    nn.Linear(128, 28)  # Final regression output (28 values)
)

# 4. (Optional) Unfreeze some layers for fine-tuning
for param in model.features[-4:].parameters():  # Last 4 layers trainable
    param.requires_grad = True

# Print the modified classifier
print(model.classifier)

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=28, bias=True)
)


/home/gcartasso/.conda/envs/court_key_points/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/gcartasso/.conda/envs/court_key_points/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
model = model.to(device)

# Train model

In [7]:
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
epochs=20
for epoch in range(epochs):
    for i, (imgs,kps) in enumerate(train_loader):
        imgs = imgs.to(device)
        print("image loaded")
        kps = kps.to(device)
        print(f"kps loaded {kps.shape}")

        optimizer.zero_grad()
        print("zero grad")
        outputs = model(imgs)
        print(f"output generated {outputs.shape}")
        loss = criterion(outputs, kps)
        print(f"loss calculated {loss}")
        loss.backward()
        print("backward pass done")
        optimizer.step()
        print("optimizer step done")
        print(f"Epoch {epoch}, iter {i}, loss: {loss.item()}")

image loaded
kps loaded torch.Size([8, 28])
zero grad
output generated torch.Size([8, 28])
loss calculated 14501.21875


In [ ]:
torch.save(model.stat_dict(), "keypoints_model.pth")